This project is not done.

Plan: For this project, I am attempting to generate a list of Siamese CBOW sentence embeddings, and then given the list of word embeddings attempt to find the words that created the sentence embedding. The plan is to compare the runtime of a brute force approach to finding the embeddings vs a dynamic programming strategy. As I'm writing this, I am realizing I have design problems. Do I do a simple set or multiset for my sentence bags of words? How long should I allow each sentence to be? How much memory is needed to create my dynamic program? 

In [1]:
import numpy as np
import pandas as pd
import fasttext
import sklearn
import torch
import re
import sentencepiece as spm
import spacy

In [2]:
nlp = spacy.load("en_core_web_sm")

In [20]:
# Split data into sentences using spaCy, this function could take a few minutes to run.

def process_corpus_into_sentences(input_file, output_file):
    with open(input_file, 'r', encoding='utf-8') as f:
        text = f.read()
        nlp.max_length = max(nlp.max_length, len(text) + 1)
        doc = nlp(text)
      

    with open(output_file, 'w', encoding='utf-8') as f:
        for sentence in doc.sents:
            tokens = [t.text for t in sentence if not t.is_space]
            if tokens: 
                f.write(" ".join(tokens) + "\n")

    print(f'Processed corpus written to {output_file}')    

In [21]:
process_corpus_into_sentences('corpus.txt', 'sentences.txt')

Processed corpus written to sentences.txt


In [22]:
# Generate word embeddings
model = fasttext.train_unsupervised('sentences.txt', model='skipgram', dim = 50)

Read 0M words
Number of words:  6671
Number of labels: 0
Progress: 100.0% words/sec/thread:  480561 lr:  0.000000 avg.loss:  2.445681 ETA:   0h 0m 0s


In [23]:
words = model.get_words(include_freq=True)


In [24]:
pairs = list(zip(words[0], words[1]))
for i in range(20):
    print(pairs[i])

(',', np.int64(39672))
('</s>', np.int64(27017))
('.', np.int64(24926))
('the', np.int64(21888))
('to', np.int64(17317))
('and', np.int64(14762))
('of', np.int64(14753))
('a', np.int64(11521))
('“', np.int64(11015))
('”', np.int64(10972))
('I', np.int64(9223))
('in', np.int64(8377))
('was', np.int64(8183))
('that', np.int64(7874))
('her', np.int64(7153))
('he', np.int64(6042))
('it', np.int64(5933))
('had', np.int64(5876))
('you', np.int64(5749))
('his', np.int64(5475))


In [25]:
# Get list of words and their associated embeddings
word2vec = {w: model.get_word_vector(w) for w in words[0]}

In [ ]:
# Generate Siamese CBOW sentence embeddings from list of original sentences from dataset
with open("sentences.txt", "r", encoding="utf-8") as f:
    sentences = [line.rstrip("\n") for line in f if line.strip()]


for sentence in sentences:
    vector_sum = 0
    words = sentence.split()
    for word in words:
        vector_sum += word2vec[word]
    vector_sum     

Who that cares much to know the history of man , and how the mysterious mixture behaves under the varying experiments of Time , has not dwelt , at least briefly , on the life of Saint Theresa , has not smiled with some gentleness at the thought of the little girl walking forth one morning hand - in - hand with her still smaller brother , to go and seek martyrdom in the country of the Moors ?
Out they toddled from rugged Avila , wide - eyed and helpless - looking as two fawns , but with human hearts , already beating to a national idea ; until domestic reality met them in the shape of uncles , and turned them back from their great resolve .
That child - pilgrimage was a fit beginning .
Theresa ’s passionate , ideal nature demanded an epic life : what were many - volumed romances of chivalry and the social conquests of a brilliant girl to her ?
Her flame quickly burned up that light fuel ; and , fed from within , soared after some illimitable satisfaction , some object which would never 

In [10]:
# Attempt to find words that produced sentence embedding from only the list of associated words
# words and embeddings